# WM-03 · 经典 World Models（Ha & Schmidhuber 风格 · 自包含）

**V 模型**（卷积 VAE 压帧）+ **M 模型**（LSTM 预测下一 latent）+ 在「梦」里滚动。

不依赖 Atari ROM；用 **自建弹球环境**，Kaggle 上最稳、必能跑通。

In [ ]:
import math, random, os
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

assert torch.cuda.is_available()
device = torch.device('cuda')
print(torch.cuda.get_device_name(0))
WORK=Path('/kaggle/working'); OUT=WORK/'wm_classic'; OUT.mkdir(exist_ok=True)

# ---- toy env: bouncing ball ----
class BallEnv:
    def __init__(self, size=64):
        self.size=size
        self.reset()
    def reset(self):
        self.x=random.uniform(0.2,0.8); self.y=random.uniform(0.2,0.8)
        ang=random.uniform(0,2*math.pi); sp=random.uniform(0.03,0.06)
        self.vx=math.cos(ang)*sp; self.vy=math.sin(ang)*sp
        self.r=0.08
        return self.render()
    def step(self, action=0):
        # action 0..3 nudge
        if action==1: self.vx-=0.01
        if action==2: self.vx+=0.01
        if action==3: self.vy-=0.01
        if action==4: self.vy+=0.01
        self.x+=self.vx; self.y+=self.vy
        if self.x<self.r or self.x>1-self.r: self.vx*=-1; self.x=float(np.clip(self.x,self.r,1-self.r))
        if self.y<self.r or self.y>1-self.r: self.vy*=-1; self.y=float(np.clip(self.y,self.r,1-self.r))
        return self.render(), 0.0, False
    def render(self):
        s=self.size
        img=Image.new('RGB',(s,s),(15,18,28))
        d=ImageDraw.Draw(img)
        d.rectangle([0,0,s-1,s-1], outline=(60,80,120), width=2)
        cx,cy=self.x*s, self.y*s; rr=self.r*s
        d.ellipse([cx-rr,cy-rr,cx+rr,cy+rr], fill=(90,200,255))
        arr=np.asarray(img).astype(np.float32)/255.0
        return arr

def collect(n_eps=40, T=40):
    env=BallEnv(); obs=[]; acts=[]
    for _ in range(n_eps):
        o=env.reset(); obs.append(o)
        for t in range(T-1):
            a=random.randint(0,4)
            o,_,_=env.step(a)
            obs.append(o); acts.append(a)
        acts.append(0)
    x=torch.tensor(np.stack(obs), dtype=torch.float32).permute(0,3,1,2)  # N,3,H,W
    a=torch.tensor(acts, dtype=torch.long)
    return x, a

print('collecting...')
X, A = collect()
print(X.shape, A.shape)

In [ ]:
# VAE
class Encoder(nn.Module):
    def __init__(self, z=32):
        super().__init__()
        self.net=nn.Sequential(
            nn.Conv2d(3,32,4,2,1), nn.ReLU(),
            nn.Conv2d(32,64,4,2,1), nn.ReLU(),
            nn.Conv2d(64,128,4,2,1), nn.ReLU(),
            nn.Flatten(),
        )
        self.mu=nn.Linear(128*8*8, z)
        self.lv=nn.Linear(128*8*8, z)
    def forward(self, x):
        h=self.net(x); return self.mu(h), self.lv(h)

class Decoder(nn.Module):
    def __init__(self, z=32):
        super().__init__()
        self.fc=nn.Linear(z, 128*8*8)
        self.net=nn.Sequential(
            nn.ConvTranspose2d(128,64,4,2,1), nn.ReLU(),
            nn.ConvTranspose2d(64,32,4,2,1), nn.ReLU(),
            nn.ConvTranspose2d(32,3,4,2,1), nn.Sigmoid(),
        )
    def forward(self, z):
        h=self.fc(z).view(-1,128,8,8); return self.net(h)

class VAE(nn.Module):
    def __init__(self, z=32):
        super().__init__(); self.enc=Encoder(z); self.dec=Decoder(z)
    def forward(self, x):
        mu,lv=self.enc(x)
        std=(0.5*lv).exp(); eps=torch.randn_like(std)
        z=mu+std*eps
        recon=self.dec(z)
        return recon, mu, lv, z

Z=32
vae=VAE(Z).to(device)
opt=torch.optim.Adam(vae.parameters(), lr=1e-3)
Xdev=X.to(device)
for ep in range(25):
    vae.train(); perm=torch.randperm(len(Xdev))
    losses=[]
    for i in range(0, len(Xdev), 64):
        b=Xdev[perm[i:i+64]]
        recon,mu,lv,z=vae(b)
        rec=F.mse_loss(recon,b)
        kl=-0.5*torch.mean(1+lv-mu.pow(2)-lv.exp())
        loss=rec+0.01*kl
        opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
    if ep%5==0: print(f'VAE ep {ep} loss={np.mean(losses):.4f}')

with torch.no_grad():
    mu,_=vae.enc(Xdev)
Zseq=mu.cpu()
print('latents', Zseq.shape)

In [ ]:
# M model: LSTM predicts next z + next action conditioning
class MDN_LSTM(nn.Module):
    def __init__(self, z=32, a=5, h=128):
        super().__init__()
        self.embed=nn.Embedding(a, 16)
        self.lstm=nn.LSTM(z+16, h, batch_first=True)
        self.out=nn.Linear(h, z)
    def forward(self, z_seq, a_seq, h=None):
        # z_seq: B,T,Z  a_seq: B,T
        ae=self.embed(a_seq)
        x=torch.cat([z_seq, ae], -1)
        y,h=self.lstm(x, h)
        return self.out(y), h

# build sequences length 20
T=20
zs=[]; as_=[]
N=len(Zseq)
for i in range(0, N-T, T):
    zs.append(Zseq[i:i+T]); as_.append(A[i:i+T])
zs=torch.stack(zs); as_=torch.stack(as_)
print('seq', zs.shape)

m=MDN_LSTM(Z,5,128).to(device)
optm=torch.optim.Adam(m.parameters(), lr=1e-3)
zs_d, as_d = zs.to(device), as_.to(device)
for ep in range(40):
    m.train()
    # predict z_{t+1} from z_t,a_t
    pred,_=m(zs_d[:,:-1], as_d[:,:-1])
    loss=F.mse_loss(pred, zs_d[:,1:])
    optm.zero_grad(); loss.backward(); optm.step()
    if ep%10==0: print(f'M ep {ep} loss={loss.item():.4f}')

In [ ]:
# Dream: start from real z0, roll M + decode V
vae.eval(); m.eval()
env=BallEnv()
real=[]
o=env.reset(); real.append(o)
for t in range(40):
    o,_,_=env.step(random.randint(0,4)); real.append(o)

with torch.no_grad():
    x0=torch.tensor(real[0], device=device).permute(2,0,1).unsqueeze(0)
    mu,_=vae.enc(x0)
    z=mu
    h=None
    dream=[]
    for t in range(40):
        img=vae.dec(z)[0].permute(1,2,0).cpu().numpy()
        dream.append((img*255).clip(0,255).astype(np.uint8))
        a=torch.tensor([[random.randint(0,4)]], device=device)
        z_in=z.unsqueeze(1); 
        pred,h=m(z_in, a, h)
        z=pred[:,0]

# real strip
real_u=[(r*255).astype(np.uint8) for r in real[:40]]
import imageio.v2 as imageio
imageio.mimsave(OUT/'classic_real.gif', real_u, fps=12, loop=0)
imageio.mimsave(OUT/'classic_dream.gif', dream, fps=12, loop=0)

# side by side sheet
def sheet(frames, n=8):
    idxs=np.linspace(0,len(frames)-1,n,dtype=int)
    ims=[Image.fromarray(frames[i]).resize((64,64)) for i in idxs]
    s=Image.new('RGB',(64*n,64))
    for i,im in enumerate(ims): s.paste(im,(i*64,0))
    return s
s1=sheet(real_u); s2=sheet(dream)
both=Image.new('RGB',(s1.width, s1.height*2+8), (10,10,10))
both.paste(s1,(0,0)); both.paste(s2,(0,s1.height+8))
both.save(OUT/'classic_real_vs_dream.png')
print('saved', list(OUT.iterdir()))
try:
    from IPython.display import display, Image as IImage
    display(IImage(filename=str(OUT/'classic_dream.gif')))
    display(both)
except Exception: pass
import shutil
shutil.make_archive('/kaggle/working/wm_classic_export','zip', OUT)
print('03 CLASSIC DONE')